# 10장 실습 ② — 학습률까지 함께 바꾼다

**TensorFlow 판**

실습 ①은 학습률을 0.001로 **고정**하고 구조만 바꿨습니다.
**5장 §5.4에서 겪은 그 함정입니다.** 확인합니다.

## 10.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 10.1 실험대 — 표시된 자리를 기억하기

순차열은 대부분 잡음입니다. **한 자리**에만 신호가 있고, 두 번째 채널이
그 자리를 표시합니다. **그 신호의 부호**를 맞힙니다.

표시된 자리가 앞쪽이므로, 그 값을 **끝까지 여러 걸음 들고 가야** 합니다.

In [ ]:
# 표시된 자리의 값을 끝까지 들고 가야 풀리는 문제.
x, y = data.memory_task(4000, length=80, seed=42)
sl = data.split(x, y, val_ratio=0.15, test_ratio=0.15, seed=42)
print(sl.summary())

fig, ax = plt.subplots(figsize=(8.5, 2.6))
ax.plot(x[0, :, 0], lw=1.2, label="값")
ax.plot(x[0, :, 1], lw=1.2, ls="--", label="표시")
ax.set_xlabel("걸음"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_title(f"표시된 자리의 부호를 맞혀야 합니다 (정답 {y[0]})")
plt.show()

## 10.2 학습 함수 — 여기만 판마다 다릅니다

**PyTorch 판에 `Recurrent` 래퍼가 하나 더 있는 것**에 주목하십시오.
PyTorch의 순환 층은 (출력 전체, 마지막 상태)를 돌려주므로,
Keras의 기본 동작(마지막 것만)과 맞추려면 감싸야 합니다.

In [ ]:
import tensorflow as tf

L_ = tf.keras.layers

def _recurrent(kind, units=32):
    return {"rnn": L_.SimpleRNN, "lstm": L_.LSTM, "gru": L_.GRU}[kind](units)

def train_seq(kind, sp, length, lr=0.001, seed=42, epochs=25):
    """분류: 표시된 자리의 부호 맞히기. (시험 정확도)

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    ls = [L_.Input(shape=(length, 2))]
    if kind == "dnn":
        ls += [L_.Flatten(), L_.Dense(64, activation="relu")]
    else:
        ls += [_recurrent(kind)]
    ls += [L_.Dense(2, activation="softmax")]
    m = tf.keras.Sequential(ls)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss="sparse_categorical_crossentropy")
    m.fit(sp.x_train, sp.y_train, epochs=dlbook.smoke.epochs(epochs),
          batch_size=64, verbose=0)
    return metrics.accuracy(sp.y_test, m.predict(sp.x_test, verbose=0).argmax(1))

def train_forecast(kind, sp, lr=0.003, seed=42, epochs=30):
    """회귀: 다음 값 맞히기. (MAE, 파라미터 수)"""
    dlbook.set_seed(seed)
    ls = [L_.Input(shape=(sp.x_train.shape[1], 1))]
    if kind == "dnn":
        ls += [L_.Flatten(), L_.Dense(64, activation="relu")]
    else:
        ls += [_recurrent(kind)]
    ls += [L_.Dense(1)]
    m = tf.keras.Sequential(ls)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="mse")
    m.fit(sp.x_train, sp.y_train, validation_data=(sp.x_val, sp.y_val),
          epochs=dlbook.smoke.epochs(epochs), batch_size=64, verbose=0)
    return metrics.mae(sp.y_test, m.predict(sp.x_test, verbose=0).reshape(-1)), \
        m.count_params()

## 10.4 실험 ② — 학습률까지 함께

실험 ①은 학습률을 고정했습니다. **5장 §5.4에서 겪은 그 함정입니다.**

In [ ]:
# 실험 ② — 길이 80으로 고정하고, 구조와 학습률을 둘 다 바꾼다.
L = 80
lrs = [0.001, 0.003] if dlbook.smoke.is_smoke() else [0.0003, 0.001, 0.003, 0.01]

print(f"길이 {L}. 구조 × 학습률.")
print(f"{'':<12}" + "".join(f"{'lr=' + str(l):>11}" for l in lrs) + f"{'최고':>9}")
for k in ("rnn", "lstm", "gru"):
    row = [train_seq(k, sl, L, lr=lr, seed=42) for lr in lrs]
    print(f"{k:<12}" + "".join(f"{v:>11.3f}" for v in row) + f"{max(row):>9.3f}")
    dlbook.record(f"ch10_best_{k}_over_lr", max(row))

print()
print("→ SimpleRNN도 LSTM도 **되는 학습률이 서로 다를 뿐** 둘 다 1.000을 냅니다.")
print("→ 학습률 세 배 차이로 0.46과 1.00이 갈립니다.")

## 정리

- **SimpleRNN도 LSTM도 되는 학습률이 서로 다를 뿐 둘 다 1.000을 냅니다.**
- **학습률 세 배 차이로 0.46과 1.00이 갈립니다.**
- 실험 ①의 결론은 **학습률을 잘못 고른 결과**였습니다.

**→ 그런데 아직 끝이 아닙니다. `ch10_seed.ipynb` 를 보십시오.**

### 연습

1. 각 구조의 "되는 학습률" 구간을 더 촘촘히 찾아보십시오.
2. `clipnorm=1.0` 을 켜면 그 구간이 넓어집니까.